**Problem 1 - Bot-aware sessionization + week-over-week retention**

In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HW2")
    .master("local[*]")
    .getOrCreate()
)

spark

In [7]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType
import re

In [11]:
event = spark.read.json("data/p1_raw_events.jsonl")
event.printSchema()
event.show(5, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- page: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- user_id: string (nullable = true)

+--------+----------+--------------------+--------------------+----------------------+------------------------------------------------------------+-------+
|event_id|event_name|event_ts            |ingested_at         |page                  |user_agent                                                  |user_id|
+--------+----------+--------------------+--------------------+----------------------+------------------------------------------------------------+-------+
|E200000 |page_view |2026-02-25T00:06:43Z|2026-02-25T00:10:41Z|/home                 |Mozilla/5.0 (X11; Linux x86_64) Gecko/20100101 Firefox/122.0|U00000 |
|E200001 |click     |2026-02-25T00:11:11Z|2026-02-25T00:13:54Z|/help                 |Mo

In [12]:
@F.udf(StringType())
def classify(agent):
    if agent is None:
        return "OTHER"

    s = agent.strip()
    if s == "":
        return "OTHER"

    s_lower = s.lower()

    if("robotics" not in s_lower) and ("reboot" not in s_lower) and ("botanist" not in s_lower):

        if re.search(r'(?i)\bbot\b', s):
            return "BOT"
        if re.search(r'(?i)\bcrawler\b', s):
            return "BOT"
        if re.search(r'(?i)\bspider\b', s):
            return "BOT"
        if re.search(r'(?i)\bslurp\b', s):
            return "BOT"
        if re.match(r'(?i)^\s*curl/', s):
            return "BOT"
        if re.match(r'(?i)^\s*wget/', s):
            return "BOT"
        if re.match(r'(?i)^\s*python-requests/', s):
            return "BOT"

        if "headlesschrome" in s_lower:
            return "BOT"

    if "ipad" in s_lower:
        return "TABLET"

    if("android" in s_lower) and ("mobile" not in s_lower):
        return "TABLET"

    if "iphone" in s_lower:
        return "MOBILE"
    if ("android" in s_lower) and ("mobile" in s_lower):
        return "MOBILE"

    if ("windows nt" in s_lower) or ("macintosh" in s_lower) or ("x11; linux" in s_lower):
        return "DESKTOP"
    
    return "OTHER"

    

In [13]:
dedup_w = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("ingested_at").desc_nulls_last(),
        F.col("event_ts").desc_nulls_last(),
        F.col("event_name").asc_nulls_last(),
        F.col("page").asc_nulls_last()
    )
)

event_dedup = (
    event
    .withColumn("rn_dedup", F.row_number().over(dedup_w))
    .filter(F.col("rn_dedup") == 1)
    .drop("rn_dedup")
)

In [16]:
events_with_ua = (
    event_dedup
    .withColumn("ua_type", classify(F.col("user_agent")))
)

events_nonbot = events_with_ua.filter(F.col("ua_type") != "BOT")

events_local = (
    events_nonbot
    .withColumn("local_ts", F.from_utc_timestamp(F.col("event_ts"), "America/New_York"))
    .withColumn("local_date", F.to_date("local_ts"))
)

In [ ]:
!git add .
!git commit -m "